In [6]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import joblib
import os

In [7]:
def train_intent_model():
    print("Membaca dataset chat_dataset.csv...")
    dataset_path = "data/chat_dataset.csv"
    
    if not os.path.exists(dataset_path):
        print("Dataset tidak ditemukan! Tunggu generate.py selesai dulu ya.")
        return None
        
    df = pd.read_csv(dataset_path)
    
    df = df.dropna()
    
    X = df['teks_chat']
    y = df['label_intent']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    print(f"Total data latih: {len(X_train)} | Total data uji: {len(X_test)}")
    
    print("Melatih model NLU (TF-IDF + SVM)...")
    model = make_pipeline(TfidfVectorizer(), SVC(kernel='linear', probability=True))
    
    model.fit(X_train, y_train)
    
    print("\n--- Hasil Ujian Model (Evaluasi) ---")
    y_pred = model.predict(X_test)
    print(classification_report(y_test, y_pred))
    
    os.makedirs("models", exist_ok=True)
    joblib.dump(model, "models/intent_classifier.pkl")
    print("Model berhasil disimpan di models/intent_classifier.pkl\n")
    
    return model

In [20]:
import os
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report

def train_intent_model_nb():
    print("Membaca dataset chat_dataset2.csv...")
    
    # Menggunakan absolute path
    base_dir = r"C:\Users\andyc\Documents\a_skripsi\training\prethesis\generateDataset"
    dataset_path = os.path.join(base_dir, "data", "chat_dataset2.csv")
    model_dir = os.path.join(base_dir, "models")
    
    if not os.path.exists(dataset_path):
        print(f"Dataset tidak ditemukan di: {dataset_path}")
        return None
        
    df = pd.read_csv(dataset_path)
    df = df.dropna()
    
    X = df['teks_chat']
    y = df['label_intent']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"Total data latih: {len(X_train)} | Total data uji: {len(X_test)}")
    
    print("Melatih model NLU (TF-IDF + Naive Bayes)...")
    model = make_pipeline(TfidfVectorizer(), MultinomialNB())
    model.fit(X_train, y_train)
    
    print("\n--- Hasil Uji Model Naive Bayes ---")
    y_pred = model.predict(X_test)
    print(classification_report(y_test, y_pred))
    
    # Pastikan folder models juga dibuat di tempat yang benar
    os.makedirs(model_dir, exist_ok=True)
    model_path = os.path.join(model_dir, "intent_classifier_nb.pkl")
    joblib.dump(model, model_path)
    print(f"Model berhasil disimpan di {model_path}\n")
    
    return model

if __name__ == "__main__":
    train_intent_model_nb()

Membaca dataset chat_dataset2.csv...
Total data latih: 650 | Total data uji: 163
Melatih model NLU (TF-IDF + Naive Bayes)...

--- Hasil Uji Model Naive Bayes ---
              precision    recall  f1-score   support

    accusing       0.46      0.52      0.49        25
    bluffing       0.52      0.78      0.62        18
    claiming       0.93      0.70      0.80        20
   defending       0.35      0.46      0.40        13
  deflecting       0.44      0.65      0.52        17
     neutral       0.95      0.75      0.84        24
  persuading       0.45      0.19      0.27        26
     probing       0.86      0.90      0.88        20

    accuracy                           0.61       163
   macro avg       0.62      0.62      0.60       163
weighted avg       0.63      0.61      0.60       163

Model berhasil disimpan di C:\Users\andyc\Documents\a_skripsi\training\prethesis\generateDataset\models\intent_classifier_nb.pkl



In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer
)

def train_intent_model_transformer():
    print("Membaca dataset chat_dataset2.csv...")
    base_dir = r"C:\Users\andyc\Documents\a_skripsi\training\prethesis\generateDataset"
    dataset_path = os.path.join(base_dir, "data", "chat_dataset2.csv")
        
    if not os.path.exists(dataset_path):
        print(f"Dataset tidak ditemukan di: {dataset_path}")
        return None

    df = pd.read_csv(dataset_path).dropna()
    
    # 1. Encode Label String (misal: 'accusing') menjadi Angka (misal: 0)
    label_encoder = LabelEncoder()
    df['label'] = label_encoder.fit_transform(df['label_intent'])
    num_labels = len(label_encoder.classes_)
    
    # 2. Split Data
    df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
    print(f"Total data latih: {len(df_train)} | Total data uji: {len(df_test)}")
    
    # 3. Ubah ke format Hugging Face Dataset
    train_dataset = Dataset.from_pandas(df_train[['teks_chat', 'label']])
    test_dataset = Dataset.from_pandas(df_test[['teks_chat', 'label']])
    
    # 4. Load Tokenizer IndoBERT
    model_name = "indobenchmark/indobert-base-p1"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    def tokenize_function(examples):
        return tokenizer(examples["teks_chat"], padding="max_length", truncation=True, max_length=128)
    
    train_dataset = train_dataset.map(tokenize_function, batched=True)
    test_dataset = test_dataset.map(tokenize_function, batched=True)
    
    # 5. Load Model Transformer
    print("Mengunduh/Memuat model IndoBERT...")
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)
    
    # 6. Konfigurasi Pelatihan (Training Arguments)
    training_args = TrainingArguments(
        output_dir="./models/transformer_results",
        eval_strategy="epoch",  # Evaluasi setiap akhir epoch
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=3,      # Jumlah epoch (bisa dinaikkan jika dataset sedikit)
        weight_decay=0.01,
        logging_dir='./logs',
    )
    
    # Buat fungsi untuk menghitung metrik evaluasi
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        predictions = np.argmax(logits, axis=-1)
        return {"accuracy": (predictions == labels).mean()}
    
    # 7. Mulai Training
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        compute_metrics=compute_metrics
    )
    
    print("Mulai melatih model Transformer...")
    trainer.train()
    
    # 8. Evaluasi & Classification Report
    print("\n--- Hasil Uji Model Transformer ---")
    predictions = trainer.predict(test_dataset)
    y_pred = np.argmax(predictions.predictions, axis=-1)
    y_true = test_dataset["label"]
    
    # Kembalikan angka ke nama intent aslinya saat di-print
    target_names = label_encoder.classes_
    print(classification_report(y_true, y_pred, target_names=target_names))
    
    # 9. Simpan Model & Tokenizer
    model_save_path = "models/intent_classifier_transformer"
    tokenizer.save_pretrained(model_save_path)
    model.save_pretrained(model_save_path)
    
    # Simpan juga label encodernya agar nanti bisa dipakai saat prediksi
    joblib.dump(label_encoder, f"{model_save_path}/label_encoder.pkl")
    
    print(f"Model berhasil disimpan di folder: {model_save_path}\n")
    return model, tokenizer, label_encoder

train_intent_model_transformer()


c:\Users\andyc\Documents\a_skripsi\training\prethesis\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Membaca dataset chat_dataset2.csv...
Total data latih: 650 | Total data uji: 163


Map: 100%|██████████| 163/163 [00:00<00:00, 10879.39 examples/s]


Mengunduh/Memuat model IndoBERT...


[transformers] You passed `num_labels=8` which is incompatible to the `id2label` map of length `5`.


In [9]:
def predict_intent(chat_text):
    model_path = "models/intent_classifier.pkl"
    if not os.path.exists(model_path):
        model = train_intent_model()
    else:
        model = joblib.load(model_path)
    
    # Prediksi intent dari chat baru
    prediksi = model.predict([chat_text])[0]
    
    # Ambil nilai probabilitas/keyakinan model (dalam persentase)
    probabilitas = max(model.predict_proba([chat_text])[0]) * 100
    
    return prediksi, probabilitas

In [28]:
train_intent_model_transformer()
# train_intent_model_nb()
# train_intent_model()
    
# print("--- SIMULASI TESTING AI 1 DI DALAM GAME ---")
# test_chats = [
#     "Jagain rumah gw pak pol, gw bayar mahal nih pake koin",
#     "Lu curigaan mulu sama gw anjir, gw cuma warga biasa",
#     "Woy si Budi dari tadi diem aja, fix dia ketuanya"
# ]

# for chat in test_chats:
#     intent, prob = predict_intent(chat)
#     print(f"Chat Player: '{chat}'")
#     print(f" > AI 1 Menebak: [{intent.upper()}] (Tingkat Keyakinan: {prob:.2f}%)\n")

Membaca dataset chat_dataset2.csv...
Total data latih: 650 | Total data uji: 163


c:\Users\andyc\Documents\a_skripsi\training\prethesis\venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\andyc\.cache\huggingface\hub\models--indobenchmark--indobert-base-p1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Map: 100%|██████████| 163/163 [00:00<00:00, 10126.97 examples/s]


Mengunduh/Memuat model IndoBERT...


ImportError: 
AutoModelForSequenceClassification requires the PyTorch library but it was not found in your environment. Check out the instructions on the
installation page: https://pytorch.org/get-started/locally/ and follow the ones that match your environment.
Please note that you may need to restart your runtime after installation.
